# 第5回 学習のしくみ — 勾配降下法と、過学習とのたたかい

**このノートブックは第5回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 勾配降下法で谷を下る(学習率を変えて、発散を目撃する)
2. 過学習を再現し、訓練誤差と検証誤差を1枚のグラフに重ねる
3. 4つの対策の効果を測って比べる
4. データを増やすと何が起きるかを確かめる

所要時間:実習 約25分 / 課題 約40分

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。

In [ ]:
# 準備(このセルを最初に1回だけ実行)
!pip install -q japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt, japanize_matplotlib
plt.rcParams["font.size"] = 12
print("準備できました / PyTorch", torch.__version__)

## 1. 勾配降下法 — 坂を下る

学習とは「損失を小さくする重みを探すこと」でした。その探しかたが **勾配降下法** です。

> **新しい重み = いまの重み − 学習率 × 傾き**

これだけです。霧の中で、足元の傾きだけを頼りに山を下りるのと同じやり方。
ここでは分かりやすさのため、重み1個・谷が1つだけの単純な例で試します。

In [ ]:
# ===== ① 勾配降下法:谷を下る =====
# 損失 L = (x - 3)^2、その傾きは 2(x - 3)。谷の底は x = 3。
def 傾き(x):
    return 2 * (x - 3)

def 降下(学習率, 歩数=20, 出発点=10.0):
    """1歩ずつ下って、通った道を全部返す"""
    x = 出発点
    道 = [x]
    for _ in range(歩数):
        x = x - 学習率 * 傾き(x)     # ← これが勾配降下法のすべて
        道.append(x)
    return 道

for lr in [0.01, 0.1, 0.5, 0.9, 1.1]:
    道 = 降下(lr)
    最初 = " → ".join(f"{v:.2f}" for v in 道[:6])
    print(f"学習率 {lr:<5} {最初} ...   20歩後 x = {道[-1]:.2f}")

In [ ]:
# 図で見る:歩幅で運命が変わる
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# 左:x の動き
for lr, col in zip([0.01, 0.1, 0.5, 0.9], ["#A8AECC", "#5B6BD6", "#3A46A8", "#E08A2B"]):
    axes[0].plot(降下(lr), "o-", ms=4, label=f"学習率 {lr}", color=col)
axes[0].axhline(3, ls="--", c="gray", lw=1)
axes[0].text(15, 3.3, "谷 (x=3)", color="gray")
axes[0].set_xlabel("歩数"); axes[0].set_ylabel("x")
axes[0].set_title("うまくいく学習率"); axes[0].legend(); axes[0].grid(alpha=.3)

# 右:発散する例だけ別枠(縦軸のスケールが違いすぎるため)
axes[1].plot(降下(1.1), "o-", ms=4, color="#E08A2B", label="学習率 1.1")
axes[1].axhline(3, ls="--", c="gray", lw=1)
axes[1].set_xlabel("歩数"); axes[1].set_ylabel("x")
axes[1].set_title("発散する学習率(縦軸に注目)"); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

print("学習率 1.1 だけ、縦軸の目盛りが桁違いになっているはずです。")
print("谷を挟んで左右に振れながら、振れ幅が毎回大きくなる ── これが発散です。")

### 発散の境目はどこか

学習率 0.9 は行きすぎるのに収束し、1.1 は発散しました。境目はどこでしょうか。

In [ ]:
# 発散するかどうかの境目を探す
print(f"{'学習率':>8}{'20歩後の x':>14}{'':>4}判定")
for lr in [0.90, 0.95, 0.99, 1.00, 1.01, 1.05, 1.10]:
    最後 = 降下(lr, 歩数=20)[-1]
    判定 = "収束" if abs(最後 - 3) < 1 else ("その場で振動" if abs(最後 - 3) < 8 else "発散")
    print(f"{lr:>8.2f}{最後:>14.2f}    {判定}")

print()
print("境目はちょうど 1.0 です。")
print("1.0 では 10 → -4 → 10 → -4 … と、同じ幅で永久に往復し続けます(近づきも遠ざかりもしない)。")

> **実務での現れかた**:学習率が大きすぎると、損失が突然 `nan`(非数)になって学習が壊れます。
> `nan` を見たら、まず学習率を下げてください。

## 2. 過学習 — 学習しすぎると悪くなる

ここからが今日の山場です。

25個の点に、大きめのモデルを当てはめます。点には**ノイズ**が混ぜてあります。
モデルがノイズまで覚えてしまうと、新しいデータでは役に立たなくなります。

In [ ]:
# ===== ② 過学習を再現する =====
# sin の曲線に、少しノイズを混ぜた点を作ります。
# 「本当の形」は sin ですが、モデルにはノイズ込みの点しか見せません。
乱数 = np.random.default_rng(5)

def データを作る(個数, seed乱数):
    x = np.sort(seed乱数.uniform(-3, 3, 個数)).astype(np.float32)
    y = (np.sin(x) + seed乱数.normal(0, 0.35, 個数)).astype(np.float32)
    return torch.tensor(x).unsqueeze(1), torch.tensor(y).unsqueeze(1)

訓練X, 訓練y = データを作る(25, 乱数)      # 学習に使う。たった25点
検証X, 検証y = データを作る(300, 乱数)     # 一度も学習に使わない

plt.figure(figsize=(7, 4))
なめらか = np.linspace(-3, 3, 200)
plt.plot(なめらか, np.sin(なめらか), "--", c="gray", lw=2, label="本当の形 (sin)")
plt.scatter(訓練X, 訓練y, s=60, color="#3A46A8", zorder=3, label="訓練データ(25点)")
plt.legend(); plt.grid(alpha=.3); plt.title("モデルが見るのは、青い25点だけ")
plt.tight_layout(); plt.show()

print("青い点は、灰色の線からノイズのぶんだけズレています。")
print("そのズレは「たまたま」であって、覚えるべきものではありません。")

In [ ]:
# 学習させながら、2つの誤差を記録する
def 学習する(隠れ=128, 層=2, 回数=3000, 学習率=0.003,
             正則化=0.0, ドロップアウト=0.0, seed=0,
             訓練X=訓練X, 訓練y=訓練y):
    torch.manual_seed(seed)
    部品 = [nn.Linear(1, 隠れ), nn.ReLU()]
    for _ in range(層 - 1):
        部品 += [nn.Linear(隠れ, 隠れ), nn.ReLU()]
        if ドロップアウト:
            部品 += [nn.Dropout(ドロップアウト)]
    部品 += [nn.Linear(隠れ, 1)]
    モデル = nn.Sequential(*部品)

    最適化 = torch.optim.Adam(モデル.parameters(), lr=学習率, weight_decay=正則化)
    損失関数 = nn.MSELoss()
    訓練履歴, 検証履歴 = [], []

    for 回 in range(回数):
        モデル.train()
        最適化.zero_grad()
        損失 = 損失関数(モデル(訓練X), 訓練y)
        損失.backward()          # ← 誤差逆伝播。この1行で全部の傾きが求まる
        最適化.step()            # ← 勾配降下法で重みを動かす
        if 回 % 10 == 0:
            モデル.eval()
            with torch.no_grad():
                訓練履歴.append(float(損失))
                検証履歴.append(float(損失関数(モデル(検証X), 検証y)))
    return モデル, np.array(訓練履歴), np.array(検証履歴)

モデル, 訓練履歴, 検証履歴 = 学習する()
底 = int(np.argmin(検証履歴))

print(f"訓練データでの誤差 : {訓練履歴[0]:.3f} → {訓練履歴[-1]:.4f}   (下がり続ける)")
print(f"見たことのないデータでの誤差 : {検証履歴[0]:.3f} → 最小 {検証履歴[底]:.3f}"
      f"(学習{底*10}回目) → 最後 {検証履歴[-1]:.3f}")
print(f"→ 検証誤差は {底*10}回目で底を打ち、そこから {(検証履歴[-1]/検証履歴[底]-1)*100:.0f}% 悪化しました。")

In [ ]:
# 今日いちばん見てほしいグラフ
回数軸 = np.arange(len(訓練履歴)) * 10

plt.figure(figsize=(9, 4.6))
plt.plot(回数軸, 訓練履歴, label="訓練データでの誤差", color="#3A46A8", lw=2)
plt.plot(回数軸, 検証履歴, label="見たことのないデータでの誤差", color="#E08A2B", lw=2)
plt.axvline(底 * 10, ls="--", c="crimson", lw=1.5)
plt.scatter([底 * 10], [検証履歴[底]], s=120, color="crimson", zorder=5)
plt.annotate(f"ここが底({底*10}回目)\nここで止めるのが正解",
             xy=(底 * 10, 検証履歴[底]), xytext=(700, 0.35),
             arrowprops=dict(arrowstyle="->", color="crimson"), color="crimson")
plt.xlabel("学習した回数"); plt.ylabel("誤差(小さいほど良い)")
plt.title("学習を続けるほど良くなる、わけではない")
plt.legend(); plt.grid(alpha=.3); plt.ylim(0, 0.6)
plt.tight_layout(); plt.show()

print("青い線(訓練誤差)だけ見ていたら、「順調」としか思えません。")
print("オレンジの線を見て、はじめて悪化に気づけます。")

### モデルが何を描いたのかを見る

数字だけだとピンと来ないので、実際に描かれた曲線を重ねてみます。

In [ ]:
# モデルが何を描いたのか、目で見る
なめらかX = torch.tensor(np.linspace(-3, 3, 300).astype(np.float32)).unsqueeze(1)
早期, _, _ = 学習する(回数=底 * 10 + 1)          # 底で止めたモデル

plt.figure(figsize=(9, 4.4))
plt.plot(なめらかX.numpy(), np.sin(なめらかX.numpy()), "--", c="gray", lw=2, label="本当の形")
plt.scatter(訓練X, 訓練y, s=55, color="#3A46A8", zorder=3, label="訓練データ")
with torch.no_grad():
    plt.plot(なめらかX.numpy(), 早期(なめらかX).numpy(), color="#02A96A", lw=2.2,
             label=f"{底*10}回で止めたモデル")
    plt.plot(なめらかX.numpy(), モデル(なめらかX).numpy(), color="#E08A2B", lw=2.2,
             label="3000回まで学習したモデル")
plt.legend(fontsize=10); plt.grid(alpha=.3)
plt.title("学習しすぎると、点を追いかけて曲がりくねる")
plt.tight_layout(); plt.show()

print("オレンジ(3000回)は、青い点をなぞろうとしてグネグネしています。")
print("緑(早期終了)のほうが、灰色の破線 ── 本当の形 ── に近いはずです。")

## 3. 4つの対策

- **早期終了** … 検証誤差が底を打った時点で止める
- **正則化** … 重みを大きくしすぎないよう制約をかける
- **ドロップアウト** … 学習のたびにランダムに一部を休ませる
- **モデルを小さくする** … そもそも覚える余地を減らす

同じデータ・同じ条件で、効果を測って比べます。

In [ ]:
# ===== ③ 4つの対策を比べる =====
結果 = [("何もしない", 検証履歴[-1]), ("早期終了(底で止める)", 検証履歴[底])]

_, _, v = 学習する(正則化=0.01)
結果.append(("正則化 weight_decay=0.01", v[-1]))
_, _, v = 学習する(ドロップアウト=0.2)
結果.append(("ドロップアウト 0.2", v[-1]))
_, _, v = 学習する(隠れ=8, 層=1)
結果.append(("モデルを小さくする(8×1層)", v[-1]))

結果 = sorted(結果, key=lambda r: r[1])
print(f"{'対策':<30}検証誤差   (小さいほど良い)")
print("-" * 52)
for 名前, 値 in 結果:
    print(f"{名前:<30}{値:.3f}   {'■' * int(値 * 60)}")

plt.figure(figsize=(8, 3.6))
plt.barh([r[0] for r in 結果][::-1], [r[1] for r in 結果][::-1], color="#3A46A8")
plt.xlabel("検証誤差(小さいほど良い)"); plt.title("対策の効果")
plt.grid(alpha=.3, axis="x"); plt.tight_layout(); plt.show()

## 4. いちばん確実な対策

上の4つとは別に、もっと確実な方法があります。**データを増やすこと**です。
モデルの大きさは変えずに、訓練データの数だけ変えて確かめます。

In [ ]:
# ===== ④ いちばん効く対策:データを増やす =====
# モデルの大きさは一切変えません。訓練データの数だけ変えます。
# 少ないデータは「たまたま」に左右されるので、3回ずつ試して平均をとります。
print(f"{'訓練データ数':>12}{'訓練誤差':>12}{'検証誤差':>12}{'開き':>10}   (3回の平均)")
print("-" * 58)
記録 = []
for 個数 in [25, 50, 100, 200]:
    訓練側, 検証側 = [], []
    for 試行 in range(3):
        乱数2 = np.random.default_rng(100 + 試行)
        tX, ty = データを作る(個数, 乱数2)
        _, tr, va = 学習する(訓練X=tX, 訓練y=ty, seed=試行)
        訓練側.append(tr[-1]); 検証側.append(va[-1])
    tr平均, va平均 = np.mean(訓練側), np.mean(検証側)
    記録.append((個数, tr平均, va平均))
    print(f"{個数:>12}{tr平均:>12.4f}{va平均:>12.3f}{va平均-tr平均:>10.3f}")

plt.figure(figsize=(7, 4))
plt.plot([r[0] for r in 記録], [r[1] for r in 記録], "o-", label="訓練誤差", color="#3A46A8")
plt.plot([r[0] for r in 記録], [r[2] for r in 記録], "o-", label="検証誤差", color="#E08A2B")
plt.fill_between([r[0] for r in 記録], [r[1] for r in 記録], [r[2] for r in 記録],
                 alpha=.15, color="#E08A2B")
plt.xlabel("訓練データの数"); plt.ylabel("誤差")
plt.title("データを増やすと、2本の線が近づく")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print()
print("色を塗った部分の広さ = 過学習の大きさ です。データが増えるほど狭くなります。")
print("モデルの大きさは変えていないことに注意してください。")
print("→ 過学習は「モデルが大きすぎる」のではなく「データに対して大きすぎる」ことで起きます。")
print()
print("※ 25点と50点はほぼ同じ結果になることがあります。データが少ないうちは、")
print("   どの点を引いたかの「運」の影響が大きいためです。100点を超えると差がはっきりします。")

## 5. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# 課題1:手で5歩ぶん計算する。損失 L = (w-2)^2、傾き 2(w-2)、出発点 w = 8
def 課題1(学習率, 歩数=5):
    w = 8.0
    道 = [w]
    for _ in range(歩数):
        w = w - 学習率 * 2 * (w - 2)
        道.append(w)
    return 道

for lr in [0.2, 1.2]:
    道 = 課題1(lr)
    print(f"学習率 {lr}: " + " → ".join(f"{v:.2f}" for v in 道))
    print(f"   谷(2)からの距離: " + " → ".join(f"{abs(v-2):.2f}" for v in 道))
    print()

# ここから下は自由に書きかえてください。
# 例:学習率をもっと大きく / 出発点を変える / 谷の位置を変える
print("--- 自分で決めた学習率で試す ---")
私の学習率 = 0.35          # ← ここを書きかえる
print(" → ".join(f"{v:.2f}" for v in 課題1(私の学習率, 歩数=8)))

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:勾配降下法を手で5歩ぶん計算する
損失 `L = (w − 2)²`、傾き `2(w − 2)`、出発点 `w = 8`。電卓でも Python でもかまいません。

- (a) 学習率 0.2 で5歩
- (b) 学習率 1.2 で5歩
- (c) 2つの数列を並べて、違いを説明しなさい

### 課題2:ノートブックを動かして報告する
- (a) 訓練誤差と検証誤差を重ねたグラフを貼り、検証誤差が底を打つ位置に印をつける
- (b) 4つの対策の効果を表にする
- (c) データ量を25点・50点・200点と変えたときの検証誤差の表を作る

### 課題3:次の主張を批判する(300字以上)

> 「モデルの学習を1週間回し続けたところ、損失が 0.05 まで下がりました。学習前は 2.3 でしたから、46分の1です。
> 十分に学習できたと判断し、このモデルを本番環境に投入します。」

考える材料:
- **どの損失**の話をしていますか? 訓練データでの損失なら、下がって当然ではありませんか?
- 「1週間回し続けた」は、良いことですか? 今日の実験では何回目で底を打ちましたか?
- 損失 0.05 は「良い数字」ですか? 何と比べれば良し悪しが分かりますか?
- そもそも、損失が小さいことと役に立つことは同じですか?(第3回)

**加点**:「では、本番に出す前に何を確認すべきか」まで書けると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。
うまくいかなかった結果も、なぜそうなったかを書いてあれば評価します。